# Imports

In [ ]:
import pathlib
import pandas as pd
import numpy as np

from experiments.models import (
    AutoETSForecast,
    AutoARIMAForecast,
    ARForecast
)

from experiments.utils import load_experiments_specs

# Load Experiment Specifications

In [ ]:
dataset = "ABC"

In [ ]:
# Load specifications
train_type = "local"
experiment_specs = load_experiments_specs(
    dataset=dataset,
    train_type=train_type,
)

# Train and Test Data
train_df = experiment_specs["train"]
test_df = experiment_specs["test"]

# Meta Data
meta = experiment_specs["meta"]
features_classical = meta["time_derived_feats"]
fcst_h = meta["fcst_h"]
freq = meta["freq"]
max_lag = meta["max_lag"]
seed = 123

# Classical Models

In [ ]:
np.random.seed(seed)

# Auto-ETS
auto_ets_fcst = AutoETSForecast(
    train=train_df,
    test=test_df.drop(columns=["value"]),
    fcst_h=fcst_h,
    freq=freq,
    season_length=max_lag,
)

# Auto-ARIMA
auto_arima_fcst = AutoARIMAForecast(
    train=train_df,
    test=test_df.drop(columns=["value"]),
    fcst_h=fcst_h,
    freq=freq,
    season_length=max_lag,
)

# Auto-ARIMA-X
auto_arimax_fcst = AutoARIMAForecast(
    train=train_df,
    test=test_df.drop(columns=["value"]),
    fcst_h=fcst_h,
    freq=freq,
    season_length=max_lag,
    features=features_classical,
)

# AR
ar_fcst = ARForecast(
    train=train_df,
    test=test_df.drop(columns=["value"]),
    fcst_h=fcst_h,
    freq=freq,
    p=max_lag,
)

# AR-X
arx_fcst = ARForecast(
    train=train_df,
    test=test_df.drop(columns=["value"]),
    fcst_h=fcst_h,
    freq=freq,
    p=max_lag,
    features=features_classical,
)

classical_fcsts = pd.concat(
    [
        auto_ets_fcst,
        auto_arima_fcst,
        auto_arimax_fcst,
        ar_fcst,
        arx_fcst,
    ],
    axis=0
)

# Combine Forecasts

In [ ]:
# Concatenate all forecasts into a single DataFrame
fcsts_df = pd.concat([
    classical_fcsts
], axis=0).reset_index(drop=True)

# Add actual values and dataset information
fcsts_df = pd.merge(
    fcsts_df,
    test_df[["dataset", "series_id", "date", "value"]],
    on=["series_id", "date"],
    how="inner"
)

# Save Forecasts

In [ ]:
repo_root = pathlib.Path.cwd()
while not (repo_root / "pyproject.toml").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent

result_path = repo_root / "experiments" / "runs" / "results" / train_type
result_path.mkdir(parents=True, exist_ok=True)

fcsts_df.to_csv(result_path / f"{dataset.lower()}_classical_fcsts.csv", index=False)